## 0.2.1 Transfer Learning from ImageNet
In this section, we use AlexNet on CIFAR-10 in two settings:
1. Fine-tuning the whole network
2. Feature extraction with pretrained weights

In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
from torchvision.models import alexnet, AlexNet_Weights
import wandb

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Make results more reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cuda


## Load CIFAR-10

We apply ImageNet normalization and resize all images to 224x224 so they match AlexNet's expected input.

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 256

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

transform_alexnet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_alexnet
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_alexnet
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

classes = train_dataset.classes
print(classes)

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## AlexNet model
We replace the last classifier layer so the output dimension becomes 10.

In [3]:
def build_alexnet(pretrained=False, freeze_features=False):
    weights = AlexNet_Weights.DEFAULT if pretrained else None
    model = alexnet(weights=weights)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 10)

    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
        for layer in model.classifier[:-1]:
            for p in layer.parameters():
                p.requires_grad = False

    return model

In [4]:
def train(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [5]:
def fit(model, train_loader, test_loader, criterion, optimizer, epochs, run_name):
    wandb.init(project="lab0-alexnet-cifar10", name=run_name)
    best_acc = 0.0
    best_state = None
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        train_loss, train_acc = train(model, train_loader, criterion, optimizer)
        test_loss, test_acc = validate(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        })

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
        )

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    wandb.finish()
    print(f"Best test accuracy: {best_acc:.4f}")
    return history, best_acc

## Experiment A — Fine-tuning AlexNet on CIFAR-10
Here we train the whole network end-to-end.

In [6]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4

model_ft = build_alexnet(pretrained=False, freeze_features=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ft.parameters(), lr=LEARNING_RATE)

history_ft, best_acc_ft = fit(
    model_ft, train_loader, test_loader,
    criterion, optimizer, NUM_EPOCHS,
    run_name="alexnet-finetune"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hamid-sabeti (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 01/50 | train_loss=1.6736 train_acc=0.3837 | test_loss=1.4046 test_acc=0.4786
Epoch 02/50 | train_loss=1.2307 train_acc=0.5543 | test_loss=1.0527 test_acc=0.6305
Epoch 03/50 | train_loss=1.0058 train_acc=0.6412 | test_loss=0.9060 test_acc=0.6821
Epoch 04/50 | train_loss=0.8534 train_acc=0.6996 | test_loss=0.8625 test_acc=0.6948
Epoch 05/50 | train_loss=0.7557 train_acc=0.7352 | test_loss=0.7326 test_acc=0.7462
Epoch 06/50 | train_loss=0.6632 train_acc=0.7686 | test_loss=0.6588 test_acc=0.7762
Epoch 07/50 | train_loss=0.5852 train_acc=0.7952 | test_loss=0.6722 test_acc=0.7667
Epoch 08/50 | train_loss=0.5179 train_acc=0.8194 | test_loss=0.5919 test_acc=0.7919
Epoch 09/50 | train_loss=0.4641 train_acc=0.8379 | test_loss=0.6321 test_acc=0.7896
Epoch 10/50 | train_loss=0.4183 train_acc=0.8529 | test_loss=0.5837 test_acc=0.8001
Epoch 11/50 | train_loss=0.3679 train_acc=0.8705 | test_loss=0.5632 test_acc=0.8075
Epoch 12/50 | train_loss=0.3155 train_acc=0.8897 | test_loss=0.5810 test_acc

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
test_acc,▁▄▅▅▆▇▇▇▇▇██████████████████████████████
test_loss,█▅▄▃▂▂▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃
train_acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇██████████████████████████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
test_acc,0.8385
test_loss,0.75713
train_acc,0.98794
train_loss,0.03663


Best test accuracy: 0.8416


## Experiment B — Feature extraction with pretrained AlexNet
Here we freeze the backbone and train only the final classifier layer.

In [7]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_fe = build_alexnet(pretrained=True, freeze_features=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LEARNING_RATE)

history_fe, best_acc_fe = fit(
    model_fe, train_loader, test_loader,
    criterion, optimizer, NUM_EPOCHS,
    run_name="alexnet-feature-extraction"
)

Epoch 01/50 | train_loss=0.7306 train_acc=0.7445 | test_loss=0.5540 test_acc=0.8004
Epoch 02/50 | train_loss=0.5957 train_acc=0.7898 | test_loss=0.5148 test_acc=0.8231
Epoch 03/50 | train_loss=0.5606 train_acc=0.8014 | test_loss=0.4945 test_acc=0.8269
Epoch 04/50 | train_loss=0.5464 train_acc=0.8079 | test_loss=0.4840 test_acc=0.8295
Epoch 05/50 | train_loss=0.5348 train_acc=0.8101 | test_loss=0.4789 test_acc=0.8315
Epoch 06/50 | train_loss=0.5244 train_acc=0.8138 | test_loss=0.4882 test_acc=0.8267
Epoch 07/50 | train_loss=0.5236 train_acc=0.8131 | test_loss=0.4855 test_acc=0.8307
Epoch 08/50 | train_loss=0.5195 train_acc=0.8170 | test_loss=0.4864 test_acc=0.8294
Epoch 09/50 | train_loss=0.5149 train_acc=0.8154 | test_loss=0.4714 test_acc=0.8350
Epoch 10/50 | train_loss=0.5122 train_acc=0.8187 | test_loss=0.4726 test_acc=0.8341
Epoch 11/50 | train_loss=0.5082 train_acc=0.8188 | test_loss=0.4857 test_acc=0.8309
Epoch 12/50 | train_loss=0.5109 train_acc=0.8207 | test_loss=0.4730 test_acc

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
test_acc,▁▆▆▇▆▆█▇▇▇▇▇▇▇▇▇▇█▇▇▆▆▇▇██▇▆▆▇▆▆▇█▇▇▇▇▇█
test_loss,█▅▃▂▂▂▂▁▂▁▂▁▂▂▂▂▂▂▂▁▂▃▂▂▂▃▁▂▄▃▂▃▂▁▃▃▂▂▂▂
train_acc,▁▅▆▆▆▇▇▇▇▇▇▇▇▇█▇▇▇▇▇▇▇█████████████▇████
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
test_acc,0.8355
test_loss,0.48053
train_acc,0.82938
train_loss,0.48085


Best test accuracy: 0.8375


## Brief comparison
Fine-tuning updates the whole AlexNet, so the model can adapt to CIFAR-10 more strongly.
Feature extraction keeps pretrained layers frozen, so only the new classifier learns.
That is why fine-tuning usually performs better when the target dataset is different from ImageNet.

In [8]:
print("===== Final Comparison (Accuracy) =====")
print(f"Fine-tuning best test acc: {best_acc_ft:.4f}")
print(f"Feature extraction best test acc: {best_acc_fe:.4f}")

===== Final Comparison (Accuracy) =====
Fine-tuning best test acc: 0.8416
Feature extraction best test acc: 0.8375
